# Hierarchy Simulation — Analysis & Plots

This notebook runs the simulation inline and visualises the key metrics:
- Gini coefficient of caloric intake
- Insider vs outsider share
- Mean myth counter (proxy for ritual ossification)
- Gatekeeper tenure
- Population turnover

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from model import HierarchyModel

plt.rcParams.update({"figure.figsize": (12, 4), "figure.dpi": 100})

## 1. Run three scenarios

In [ ]:
STEPS = 2000

scenarios = {
    "Baseline (flat)": dict(escape_cost=0.1, granary_capacity=0.0),
    "Moderate": dict(escape_cost=0.5, granary_capacity=50.0),
    "Hardening": dict(escape_cost=0.9, granary_capacity=200.0),
}

results = {}
for name, overrides in scenarios.items():
    print(f"Running {name} …")
    m = HierarchyModel(seed=42, **overrides)
    for _ in range(STEPS):
        m.step()
        if len(m._living_agents()) == 0:
            break
    results[name] = m.datacollector.get_model_vars_dataframe()
    print(f"  {name}: {len(results[name])} ticks recorded")

## 2. Gini coefficient over time

In [ ]:
fig, ax = plt.subplots()
for name, df in results.items():
    ax.plot(df.index, df["Gini_Intake"].rolling(50).mean(), label=name)
ax.set_xlabel("Tick")
ax.set_ylabel("Gini (50-tick rolling avg)")
ax.set_title("Caloric Inequality Over Time")
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig("output/gini_over_time.png")
plt.show()

## 3. Insider share over time

In [ ]:
fig, ax = plt.subplots()
for name, df in results.items():
    ax.plot(df.index, df["Insider_Share"].rolling(50).mean(), label=name)
ax.set_xlabel("Tick")
ax.set_ylabel("Insider share")
ax.set_title("In-group Fraction Over Time")
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig("output/insider_share.png")
plt.show()

## 4. Mean myth counter (ritual ossification proxy)

In [ ]:
fig, ax = plt.subplots()
for name, df in results.items():
    ax.plot(df.index, df["Mean_Myth"].rolling(50).mean(), label=name)
ax.set_xlabel("Tick")
ax.set_ylabel("Mean myth_counter")
ax.set_title("Narrative Reinforcement Over Time")
ax.legend()
plt.tight_layout()
plt.savefig("output/myth_counter.png")
plt.show()

## 5. Population and turnover

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for name, df in results.items():
    axes[0].plot(df.index, df["Population"], label=name)
axes[0].set_xlabel("Tick")
axes[0].set_ylabel("Population")
axes[0].set_title("Population Over Time")
axes[0].legend()

for name, df in results.items():
    turnover = (df["Deaths"] + df["Migrations"]) / df["Population"].clip(lower=1)
    axes[1].plot(df.index, turnover.rolling(50).mean(), label=name)
axes[1].set_xlabel("Tick")
axes[1].set_ylabel("Turnover rate")
axes[1].set_title("Population Turnover (deaths + exits / total)")
axes[1].legend()

plt.tight_layout()
plt.savefig("output/population.png")
plt.show()

## 6. Gatekeeper tenure

In [ ]:
fig, ax = plt.subplots()
for name, df in results.items():
    ax.plot(df.index, df["Gatekeeper_Tenure"], label=name, alpha=0.7)
ax.set_xlabel("Tick")
ax.set_ylabel("Tenure (ticks)")
ax.set_title("Gatekeeper Tenure Over Time")
ax.legend()
plt.tight_layout()
plt.savefig("output/gatekeeper_tenure.png")
plt.show()

## 7. Summary table

In [ ]:
summary = []
for name, df in results.items():
    tail = df.tail(200)  # last 200 ticks
    summary.append({
        "Scenario": name,
        "Mean Gini (last 200)": f"{tail['Gini_Intake'].mean():.3f}",
        "Mean Insider Share": f"{tail['Insider_Share'].mean():.3f}",
        "Mean Myth Counter": f"{tail['Mean_Myth'].mean():.1f}",
        "Max GK Tenure": int(df["Gatekeeper_Tenure"].max()),
        "Final Population": int(df["Population"].iloc[-1]),
    })
pd.DataFrame(summary)